In [1]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import optuna
import wandb


In [2]:
# Settings
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


STYLE_DIR = r"D:\COURSE_DATA\Intro_Deep_Learning\project\data\Post_Impressionism"
CONTENT_DIR = r"D:\COURSE_DATA\Intro_Deep_Learning\project\pics"
JUDGE_MODEL_PATH = "best_vangogh_vgg19_final.pth"

#Settings for hyper parameters ssearch
NUM_IMAGES_TO_TEST = 3
SEARCH_IMG_SIZE = 224
NUM_STEPS = 300

print(f"✅ Device: {DEVICE}")

✅ Device: cuda


In [3]:
# Painter and judge models
# ==========================================

# VGG Extractor (Painter)
print("🎨 Loading Feature Extractor...")
vgg_extractor = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features.to(DEVICE).eval()
for param in vgg_extractor.parameters():
    param.requires_grad_(False)
# Judge Model
def load_judge_model(path):
    print(f"⚖️ Loading Judge from {path}...")
    model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
    # התאמה למספר המחלקות (בהנחה שזה המודל מהחלק הקודם עם 2 מחלקות)
    model.classifier[6] = nn.Linear(4096, 2)
    try:
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        print("✅ Judge loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading judge: {e}")
        # אם אין מודל, אי אפשר להמשיך
        exit()
    model.to(DEVICE).eval()
    for param in model.parameters():
        param.requires_grad = False
    return model


if os.path.exists(JUDGE_MODEL_PATH):
    judge_model = load_judge_model(JUDGE_MODEL_PATH)
else:
    print(f"⚠️ Warning: Judge model not found at {JUDGE_MODEL_PATH}. Please make sure the file exists.")
    exit()

🎨 Loading Feature Extractor...
⚖️ Loading Judge from best_vangogh_vgg19_final.pth...
✅ Judge loaded successfully.


In [4]:
# Features and images loading
# ==========================================
def load_image_tensor(img_path, max_size=SEARCH_IMG_SIZE):
    try:
        image = Image.open(img_path).convert('RGB')
        # Converting images to uniformal dimensions
        in_transform = transforms.Compose([
            transforms.Resize((max_size, max_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
        return in_transform(image)[:3, :, :].unsqueeze(0).to(DEVICE)
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        return None


def get_image_paths(directory, is_style=False, limit=3):
    if not os.path.exists(directory): return []
    all_files = os.listdir(directory)
    valid_images = []
    for f in all_files:
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        if is_style:
            # Filtering only van gogh photos
            if "vincent-van-gogh" in f.lower():
                valid_images.append(os.path.join(directory, f))
        else:
            valid_images.append(os.path.join(directory, f))

    # Shuffling
    random.shuffle(valid_images)
    return valid_images[:limit]


def get_features(image, model, layers_dict):
    features = {}
    x = image
    for name, layer in model._modules.items():
        x = layer(x)
        if name in layers_dict:
            features[layers_dict[name]] = x
    return features


def gram_matrix(tensor):
    b, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    return torch.mm(tensor, tensor.t())

In [5]:
# Style Transfer
# ==========================================
def run_style_transfer_single(content_tensor, style_tensor, c_weight, s_weight, tv_weight, lr,
                              layer_weights_dict, content_layer_name, num_steps=NUM_STEPS):
    # Matching shapes
    if content_tensor.shape != style_tensor.shape:
        style_tensor = F.interpolate(style_tensor, size=content_tensor.shape[-2:], mode='bilinear')

    target = content_tensor.clone().requires_grad_(True).to(DEVICE)
    optimizer = optim.Adam([target], lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

    # Layers mapping
    layers = {
        '0': 'conv1_1', '5': 'conv2_1', '10': 'conv3_1', '12': 'conv3_2',
        '19': 'conv4_1', '21': 'conv4_2', '28': 'conv5_1', '30': 'conv5_2'
    }

    # Features
    content_features = get_features(content_tensor, vgg_extractor, layers)
    style_features = get_features(style_tensor, vgg_extractor, layers)
    style_grams = {layer: gram_matrix(style_features[layer]) for layer in style_features if layer in layer_weights_dict}

    for i in range(num_steps):
        target_features = get_features(target, vgg_extractor, layers)

        # 1. Content Loss
        c_loss = torch.mean((target_features[content_layer_name] - content_features[content_layer_name]) ** 2)

        # 2. Style Loss
        s_loss = 0
        for layer_name, weight in layer_weights_dict.items():
            if layer_name in target_features:
                target_gram = gram_matrix(target_features[layer_name])
                style_gram = style_grams[layer_name]

                b, d, h, w = target_features[layer_name].shape
                layer_s_loss = weight * torch.mean((target_gram - style_gram) ** 2)
                s_loss += layer_s_loss / (d * h * w)

        # 3. Total Variation Loss
        diff_i = torch.sum(torch.abs(target[:, :, :, 1:] - target[:, :, :, :-1]))
        diff_j = torch.sum(torch.abs(target[:, :, 1:, :] - target[:, :, :-1, :]))
        tv_loss = (diff_i + diff_j) / (target.nelement())

        # Loss
        total_loss = c_weight * c_loss + s_weight * s_loss + tv_weight * tv_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        scheduler.step()

    return target


In [6]:
# Data preparation
# ==========================================
print("\n--- Preparing Data for Search ---")
content_paths = get_image_paths(CONTENT_DIR, is_style=False, limit=NUM_IMAGES_TO_TEST)
style_paths = get_image_paths(STYLE_DIR, is_style=True, limit=NUM_IMAGES_TO_TEST)

if not content_paths or not style_paths:
    print("❌ Error: Not enough images found. Please check your directories.")
    exit()

# Loading to tensors
content_tensors = [load_image_tensor(p) for p in content_paths]
style_tensors = [load_image_tensor(p) for p in style_paths]
# Filtering images that loaded inappropriately
content_tensors = [t for t in content_tensors if t is not None]
style_tensors = [t for t in style_tensors if t is not None]

min_len = min(len(content_tensors), len(style_tensors))
content_tensors = content_tensors[:min_len]
style_tensors = style_tensors[:min_len]

print(f"✅ Ready on {min_len} pairs of images. Size: {SEARCH_IMG_SIZE}x{SEARCH_IMG_SIZE}")


--- Preparing Data for Search ---
✅ Ready on 3 pairs of images. Size: 224x224


In [7]:
# Objective function for optuna
# ==========================================
def objective(trial):
    # 1. Content Weight
    c_weight = 1.0

    # 2. Style Weight
    s_weight = trial.suggest_float("style_weight", 1e4, 1e9, log=True)

    # 3. Learning Rate
    lr = trial.suggest_float("lr", 0.01, 0.1)

    # 4. TV Weight
    tv_weight = trial.suggest_float("tv_weight", 1e-6, 1e-3, log=True)

    # Content layer
    content_layer_name = trial.suggest_categorical("content_layer", ['conv3_2', 'conv4_2', 'conv5_2'])

    # Style layers weights
    w1 = trial.suggest_float("w_conv1", 0.1, 1.0)
    w2 = trial.suggest_float("w_conv2", 0.1, 1.0)
    w3 = trial.suggest_float("w_conv3", 0.1, 1.0)
    w4 = trial.suggest_float("w_conv4", 0.1, 1.0)
    w5 = trial.suggest_float("w_conv5", 0.1, 1.0)

    current_layer_weights = {
        'conv1_1': w1, 'conv2_1': w2, 'conv3_1': w3, 'conv4_1': w4, 'conv5_1': w5
    }

    # Performing style transfer in images
    total_score = 0.0

    for i in range(len(content_tensors)):
        ct = content_tensors[i]
        st = style_tensors[i]

        # Generating images
        generated_img = run_style_transfer_single(
            ct, st, c_weight, s_weight, tv_weight, lr,
            layer_weights_dict=current_layer_weights,
            content_layer_name=content_layer_name,
            num_steps=NUM_STEPS
        )

        # Evaluation by the judge model
        with torch.no_grad():
            output = judge_model(generated_img)
            prob = F.softmax(output, dim=1)[0, 1].item()

        total_score += prob

    avg_score = total_score / len(content_tensors)

    # Reporting for WanDB
    wandb.log({
        "trial": trial.number,
        "score": avg_score,
        "style_weight": s_weight,
        "lr": lr,
        "tv_weight": tv_weight,
        "content_layer": content_layer_name,
        "w_conv1": w1, "w_conv2": w2, "w_conv3": w3, "w_conv4": w4, "w_conv5": w5
    })

    print(f"Trial {trial.number}: Score={avg_score:.4f} | StyleW={s_weight:.2e} | LR={lr:.3f}")

    return avg_score

In [9]:
# main
# ==========================================
if __name__ == "__main__":
    print("🚀 Starting Optuna Search with W&B Tracking...")
    wandb.init(project="vangogh-style-transfer-search", name="optuna_search_vgg", reinit=True)

    # Creating Optuna study
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30)
    print("\n" + "=" * 50)
    print("🏆 SEARCH FINISHED")
    print(f"Best Judge Score: {study.best_value:.4f}")
    print("Best Params:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    print("=" * 50)

    wandb.finish()

🚀 Starting Optuna Search with W&B Tracking...


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[I 2026-01-18 14:12:24,892] A new study created in memory with name: no-name-fa41c79f-acc4-44f7-9cd7-ac90babb77c1


Trial 0: Score=0.5609 | StyleW=9.44e+08 | LR=0.083


[I 2026-01-18 14:12:32,767] Trial 0 finished with value: 0.5609323208530744 and parameters: {'style_weight': 944034439.5185802, 'lr': 0.08258153257278483, 'tv_weight': 4.477889842264191e-06, 'content_layer': 'conv4_2', 'w_conv1': 0.64482623366814, 'w_conv2': 0.4838028046299385, 'w_conv3': 0.5154603903123034, 'w_conv4': 0.10146191207683304, 'w_conv5': 0.5621104603027242}. Best is trial 0 with value: 0.5609323208530744.


...

[I 2026-01-18 14:16:13,835] Trial 28 finished with value: 0.6433327544558173 and parameters: {'style_weight': 3643580.105949867, 'lr': 0.05132013936749561, 'tv_weight': 0.00019635794758140403, 'content_layer': 'conv4_2', 'w_conv1': 0.3383220606580524, 'w_conv2': 0.6767946529026366, 'w_conv3': 0.7947632229238626, 'w_conv4': 0.41263907978483816, 'w_conv5': 0.6480416131136386}. Best is trial 10 with value: 0.6599737361539155.


Trial 29: Score=0.6078 | StyleW=5.43e+08 | LR=0.085


[I 2026-01-18 14:16:22,060] Trial 29 finished with value: 0.6077553372209271 and parameters: {'style_weight': 542518601.9556998, 'lr': 0.08466816544649178, 'tv_weight': 9.381638244571267e-06, 'content_layer': 'conv5_2', 'w_conv1': 0.5216913650211967, 'w_conv2': 0.4920703685653869, 'w_conv3': 0.8531354076524891, 'w_conv4': 0.13758116280074, 'w_conv5': 0.5934230877891121}. Best is trial 10 with value: 0.6599737361539155.



🏆 SEARCH FINISHED
Best Judge Score: 0.6600
Best Params:
  style_weight: 112666.28998144648
  lr: 0.09983942885924048
  tv_weight: 0.000625132659617113
  content_layer: conv3_2
  w_conv1: 0.9177914760609815
  w_conv2: 0.9841844103458328
  w_conv3: 0.762457827861725
  w_conv4: 0.38811898204357387
  w_conv5: 0.9906831201129968


lr,▇▃▅▂▅▇▂▃▇▃███▆▆▇▅▁█▇▇▇▆▇▆█▆▇▄▇
score,▂▅▇▅█▆▆▆█▆█▅▄▇▇▇▁▅▃█▇▇▇█▇▆▆█▇▅
style_weight,█▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅
trial,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
tv_weight,▁▁▁▂▁▂▁▂▃▁▅█▇▃▃▄▃▁▇▄▅▂▄▁▂▄▁▅▂▁
w_conv1,▅▅▂▁▁▁▃▃▅▂▇█▇▆▆█▆▄▇▄▄▃▅▅▆▄█▅▃▄
w_conv2,▄▂▇▆▅▅▄▅▅▃██▆▂▁█▇▃▆▅▄▃▅▆▇▅▆▃▅▄
w_conv3,▄▂▄▅█▁▄█▅▄▆▆▆▆▅▂▇▅▃▇▇▆▇▅▇▆█▄▆▇
w_conv4,▁▆▅▅█▃▃▆▃█▃▃▂▄▂▅▁▄▃▅▆▅▄▅▃▂▆▄▄▁
w_conv5,▅▂▂▆▄▂▂▆▆▃███▇▅▁▇▆▇▅▄▅▇▆▆▄▆▇▅▅
content_layer,conv5_2
